<div style="text-align: center; font-family: Arial, sans-serif; color: #003366;">

<font size=6><b>[LGBIO2020] - Bioinstrumentation</b><br></font>  
<font size=6>Project EEG classification : Open your eyes! </font>  
<br>

<font size=5>Hilda Ferner</font>  
<br><br>


</div>


The complete processing pipeline is summarized as follows:
Raw EEG acquisition → initial quality inspection → band-pass filtering (0.5–40 Hz) → post-filter channel quality assessment → removal of persistent bad channel (AF7) → segmentation into 500 ms epochs with 50% overlap → artifact rejection (63 s contamination) → feature extraction (relative band power) → train-test split (80/20, temporal block-wise) → PCA dimensionality reduction → classification → performance evaluation.

<br><br>

<font size=6 color=#003366> <center><b>PART I - DATA PREPROCESSING </b></center> </font> <br> <br>


<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">
    <font size=5><b> I.1. Visualization <br></b></font>  
</div> <br>

<div class="alert alert-info">
<b>[Question 1] Import and visualize the EEG signals</b>  <br>
The EEG recordings are provided as <samp>*.csv</samp> files, each containing 64-channel signals recorded during an "eyes open" or "eyes closed" condition. Load the EEG recordings an plot the raw signals. Comment briefly on any visible patterns or differences you can observe. (You can also consider a visualization in the Fourier domain). 
</div> 

<div class="alert alert-warning">
<b>[Remark] Do not display all your plots in the notebook </b>  <br>
Especially in the early stage of the project, you will produce a lot of graphs to validate your approach. Showing all of them in the notebook can be a nightmare. Instead, consider saving them in 'png' files to inspect them, or use dedicated toolboxes.
</div> 

In [1]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Qt5Agg') #plots will not be shown in the notebook
import mne #python package for EEG processing 
import pandas as pd


#Load EEG data from CSV file 
#.values converts the dataframe into a numpy array 
closed = pd.read_csv('data/eeg_eyes_closed.csv', header=None).values
open_  = pd.read_csv('data/eeg_eyes_open.csv', header=None).values #we use "open_" since "open" is a built in python funciton  
channels = pd.read_csv('data/channels_names.csv', header=None).values.flatten()


#create EEG metadata for MNE
#ch_names: channel names, sfreq: sampling frequency, ch_types: all channels are EEG channels
fs = 1024 
info = mne.create_info(ch_names=list(channels), sfreq=fs, ch_types='eeg')

#MNE stores EEG data in raw objects with the shape (n_channels,n_times)
raw_closed = mne.io.RawArray(closed, info)
raw_open = mne.io.RawArray(open_, info)

#electrode positions
#i will use standard 10-20 placements system 
montage = mne.channels.make_standard_montage('standard_1020')
raw_closed.set_montage(montage)
raw_open.set_montage(montage)

# Plot raw data 
# duration: how many seconds to show; n_channels: how many channels to stack
#creates an interactive plot
raw_closed.plot(duration=15, n_channels=16, title="Eyes Closed Raw", scalings='auto',show=True, block=True)
raw_open.plot(duration=15, n_channels=16, title="Eyes Open Raw", scalings='auto',show=True, block=True)

# Plot PSD (Power Spectral Density), fourier domain
raw_closed.compute_psd(fmax=60).plot(picks='data',exclude='bads')
plt.gcf().suptitle("Eyes Closed")

raw_open.compute_psd(fmax=60).plot(picks='data', exclude='bads')
plt.gcf().suptitle("Eyes Open")

#plots the psd as topogrophy maps, in the frequecy bands 
raw_closed.compute_psd().plot_topomap()
raw_open.compute_psd().plot_topomap()

plt.show()


Creating RawArray with float64 data, n_channels=64, n_times=102401
    Range : 0 ... 102400 =      0.000 ...   100.000 secs
Ready.
Creating RawArray with float64 data, n_channels=64, n_times=102401
    Range : 0 ... 102400 =      0.000 ...   100.000 secs
Ready.
Using matplotlib as 2D backend.
Channels marked as bad:
none
Channels marked as bad:
none
Effective window size : 2.000 (s)
Plotting power spectral density (dB=True).
Effective window size : 2.000 (s)
Plotting power spectral density (dB=True).
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)


Comments on raw data visualiztion:\
Closed eyes\
In the time domain, the EEG signals showed pronounced activity in frontal electrodes, along with low-frequency drifts and intermittent high-frequency noise. A global artifact was also observed around specific time points, suggesting non-neural interference.
In the frequency domain, a clear increase in power was observed in the alpha band (~8–13 Hz), consistent with typical eyes-closed resting-state activity. Additional spectral components were present across theta and beta ranges, with localized gamma activity primarily in left-sided channels. A strong peak near 50 Hz indicated the presence of line noise. Overall, alpha activity appeared more prominent in posterior regions, while frontal electrodes exhibited higher broadband power and noise.

Open eyes\
When the eyes were open, the time-domain signals exhibited more bilateral frontal activity and frequent transient peaks, likely corresponding to eye blinks and muscle artifacts. Compared to the eyes-closed condition, the signal appeared more irregular with increased high-frequency components.
The PSD analysis showed a reduction in the relative prominence of the alpha band, which is consistent with known neurophysiological behavior during visual engagement. Instead, power was more evenly distributed across frequency bands, with persistent peaks around 1 Hz (slow drift) and 50 Hz (power-line interference). Topographic maps indicated stronger broadband activity in frontal regions, while occipital alpha activity was less pronounced. 





<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">
    <font size=5><b> I.2. Remove artefact <br></b></font>  
</div> <br>

<div class="alert alert-info">
<b>[Question 2] Discard irrelevant channels</b>  <br>
Based on your visual inspection of the EEG signals, are there any channels you would consider discarding from the analysis? If so, list the channels you would discard and justify your choice (Are they too noisy, flat, or showing unusual artifacts?). 
</div> 


Since visual inspection alone does not reliably distinguish bad channels especially when all channels exhibit some level of high-frequency noise and low-frequency drift I decidaaned to use an objective approach, a statistical outlier detection method based on channel variance. The underlying assumption is that “clean” EEG channels should exhibit comparable variance levels, while abnormal channels either show excessively high variance (noisy or corrupted electrodes) or unusually low variance (flat or disconnected signals). A bad channel would only be removed if it was identified as bad in both datasets. 

In [96]:
import matplotlib.pyplot as plt
import numpy as np

def detect_bad_channels(raw, threshold_z=3.0):
    
    #Identifies channels that have significantly higher or lower variancethan the average channel using Z-scores.
    # For each channels, the variance over time was computed. 
    # A z-score was then calculated across all channel variance. 
    
    # Get data from all channels
    data = raw.get_data()  # (n_channels, n_times)
    
    # Calculate the variance for every channel
    channel_variances = np.var(data, axis=1)
    
    # Calculate Z-scores for the variances
    mean_var = np.mean(channel_variances)
    std_var = np.std(channel_variances)
    z_scores = (channel_variances - mean_var) / std_var
    
    # Find channels where variance is too high or too low
    bad_indices = np.where(np.abs(z_scores) > threshold_z)[0]
    bad_names = [raw.ch_names[i] for i in bad_indices]
    
    return bad_names, z_scores

# Run detection
bads_closed, scores_closed = detect_bad_channels(raw_closed)
bads_open, scores_open = detect_bad_channels(raw_open)

# Combine and unique list
all_bads = list(set(bads_closed + bads_open))

print(f"Automatically detected bad channels: {all_bads}")

# Plot both on the same graph
plt.figure(figsize=(14, 5))

channels = np.arange(len(scores_closed))
plt.plot(channels, scores_closed, marker='o', label='Closed Eyes')
plt.plot(channels, scores_open, marker='s', label='Open Eyes')

# Threshold lines
plt.axhline(3, color='red', linestyle='--', label='Threshold ±3')
plt.axhline(-3, color='red', linestyle='--')

# Highlight detected bad channels
for i, z in enumerate(scores_closed):
    if abs(z) > 3:
        plt.scatter(i, z, color='red', s=80)

for i, z in enumerate(scores_open):
    if abs(z) > 3:
        plt.scatter(i, z, color='orange', s=80)

# Optional: show channel names instead of indices
plt.xticks(channels, raw_closed.ch_names, rotation=90)

plt.title("Z-Score of Channel Variances (Open vs Closed)")
plt.xlabel("Channels")
plt.ylabel("Z-Score")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Automatically detected bad channels: ['AF7']


<div class="alert alert-info"> <b>[Question 3] Filtering EEG signals</b> <br> Before training a classifier, it is often useful to apply signal preprocessing to remove undesired components such as slow drifts, power line noise, or muscle artifacts.<br>
If you decide filter your signals: 
<ul>
    <li>Justify your choices based on time-domain and/or frequency-domain analysis.</li>
    <li>Discuss how your filtering decisions are expected to improve the extraction of meaningful features for classification.</li>
</ul>
</div> 

From the visulaization of the raw data (in both domains) I could identify several sources of noise, including slow drifts, muscle artifacts, and power-line interference.

I decided to apply a zero-phase finite impulse response (FIR) band-pass filter with cutoff frequencies set to 0.5 Hz and 40 Hz. The high-pass filter (0.5 Hz) removes slow DC drifts and very low-frequency fluctuations, while the low-pass filter (40 Hz) reduces high-frequency noise, including muscle artifacts and power-line interference (50 Hz component).

I used the MNE function raw.filter which use firwin, with a Hamming window. 

In [97]:
# Filtering 
# High-pass at 0.5Hz to remove DC drift, Low-pass at 40Hz to remove 50Hz noise
raw_closed.filter(l_freq=0.5, h_freq=40)
raw_open.filter(l_freq=0.5, h_freq=40)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 6759 samples (6.601 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband ed

<RawArray | 64 x 102401 (100.0 s), ~50.1 MiB, data loaded>

After the filtering i decided to run the bad channels function agian, to see if the removal of low frequency drift would effect the result. 

In [98]:
# Run detection
bads_closed, scores_closed = detect_bad_channels(raw_closed)
bads_open, scores_open = detect_bad_channels(raw_open)

# Combine and unique list
all_bads = list(set(bads_closed + bads_open))

print(f"Automatically detected bad channels: {all_bads}")

# Plot both on the same graph
plt.figure(figsize=(14, 5))

channels = np.arange(len(scores_closed))

plt.plot(channels, scores_closed, marker='o', label='Closed Eyes')
plt.plot(channels, scores_open, marker='s', label='Open Eyes')

# Threshold lines
plt.axhline(3, color='red', linestyle='--', label='Threshold ±3')
plt.axhline(-3, color='red', linestyle='--')

# Highlight detected bad channels
for i, z in enumerate(scores_closed):
    if abs(z) > 3:
        plt.scatter(i, z, color='red', s=80)

for i, z in enumerate(scores_open):
    if abs(z) > 3:
        plt.scatter(i, z, color='orange', s=80)

# Optional: show channel names instead of indices
plt.xticks(channels, raw_closed.ch_names, rotation=90)

plt.title("Z-Score of Channel Variances After Filtering (Open vs Closed)")
plt.xlabel("Channels")
plt.ylabel("Z-Score")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Automatically detected bad channels: ['AF7']


Since AF7 was indicated both before and after the filtering i decided to drop that channel.

In [99]:
#remove bad channels
bad_channels = ['AF7'] 

# mark them in the metadata
raw_closed.info['bads'] = bad_channels
raw_open.info['bads'] = bad_channels

# physically remove them from the data arrays
raw_closed.drop_channels(raw_closed.info['bads'])
raw_open.drop_channels(raw_open.info['bads'])

<RawArray | 63 x 102401 (100.0 s), ~49.3 MiB, data loaded>

In [100]:
#plot the filtered data without the bad channel 
#the deafult y_axis eange makes the signals look very "messy", but change the voltage by pressing + or - 
raw_closed.plot(duration=15, n_channels=16, title="Eyes Closed Raw", scalings='auto',show=True, block=True)
raw_open.plot(duration=15, n_channels=16, title="Eyes Open Raw", scalings='auto',show=True, block=True)

raw_closed.compute_psd(fmax=60).plot(picks='data')
plt.gcf().suptitle("PSD After 0.5-40Hz Filtering - Eyes Closed")
raw_closed.compute_psd().plot_topomap()

raw_open.compute_psd(fmax=60).plot(picks='data')
plt.gcf().suptitle("PSD After 0.5-40Hz Filtering - Eyes Open")
raw_open.compute_psd().plot_topomap()

Channels marked as bad:
none
Channels marked as bad:
none
Effective window size : 2.000 (s)
Plotting power spectral density (dB=True).
Effective window size : 2.000 (s)
Effective window size : 2.000 (s)
Plotting power spectral density (dB=True).
Effective window size : 2.000 (s)


<Figure size 1000x150 with 10 Axes>

After filtering, the EEG signals were re-visualized in both time and frequency domains. In the time domain, slow drifts were significantly reduced, resulting in more stable baselines across channels. Transient artifacts such as spikes and eye blinks remained visible but were less dominant.
In the frequency domain, Power Spectral Density (PSD) analysis confirmed the expected effects of filtering. Low-frequency components below 0.5 Hz were removed, and high-frequency activity above 40 Hz was strongly attenuated. The prominent 50 Hz line noise observed in the raw data was also substantially reduced.
Topographic PSD maps further showed a clearer spatial distribution of physiologically relevant frequency bands, particularly in the alpha range, which is known to be sensitive to eye state changes.


<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">
    <font size=5><b> I.3. Divide the signals into short segments <br></b></font>  
</div> <br>

<div class="alert alert-info">
<b>[Question 4] Segment your data </b>  <br>
Divide the entire time signal into short segments of 500 ms each. Each segment will serve as a single training, validation, or test sample in your final dataset.
    
Consideration: Would it be beneficial to use overlapping segments?
</div> 

Using overlapping segments provides several benefits like increased dataset size, which will increase the number of available training samples , which is particularly beneficial for machine learning models. It can also improve robustness since neural patterns that span across window boundaries are less likely to be truncated, reducing information loss.Overlapping windows also preserve continuous brain dynamics and allow gradual transitions in rhythmic activity to be captured more effectively. 

However, overlapping segmentation also introduces some drawbacks. Consecutive epochs become highly similar, which reduces the amount of truly independent information in the dataset. More samples also increase training time and memory requirements and highly correlated samples may lead to overly optimistic performance estimates if not properly validated.

I still chose to do overlapping, mainly to create an increased dataset. Each EEG recording was divided into 0.5-second segments with a 50% overlap (i.e., 0.25-second step size). This means that consecutive windows share half of their samples.

During exploratory analysis, a strong global artifact was identified around 63 seconds in the eyes-open recording. To ensure data quality and prevent contamination of the feature space, this segment was explicitly excluded from both conditions to maintain consistency. This was implemented using MNE annotations, which allow marking time intervals as bad segments. A 2-second window centered around the artifact was labeled and automatically excluded from epoch extraction.

In [101]:
#Define parameters
segment_duration = 0.5  # Length of each window in seconds
overlap_duration = 0.25 # How much the windows share (50% overlap =0.25s)

# Create the events with the 'overlap' parameter
# This generates markers that start every 0.25 seconds, lasting 0.5 seconds each
events_closed = mne.make_fixed_length_events(raw_closed, duration=segment_duration, overlap=overlap_duration)
events_open = mne.make_fixed_length_events(raw_open, duration=segment_duration, overlap=overlap_duration)

#we need to remove the epochs that includes t=63s, since there is a global artifact there in one of the data sets, we will remove the same epochs from both datasets
annotations = mne.Annotations(onset=[61.5],duration=[1], description=["bad_artifact"]) #MNE will automatically reject annotations starting with bad 
raw_closed.set_annotations(annotations)
raw_open.set_annotations(annotations)


# Create the Epochs as before
#Epoch= short time/signal segment (not a full training cycle through data)
# MNE will automatically grab 500ms from each event start point
epochs_closed = mne.Epochs(raw_closed, events_closed, tmin=0, tmax=segment_duration - 1/fs, baseline=None, preload=True)
epochs_open = mne.Epochs(raw_open, events_open, tmin=0, tmax=segment_duration - 1/fs, baseline=None, preload=True)


print(f"New count for 'Closed': {len(epochs_closed)}")
print(f"New count for 'Open': {len(epochs_open)}")

Not setting metadata
399 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 399 events and 512 original time points ...
5 bad epochs dropped
Not setting metadata
399 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 399 events and 512 original time points ...
5 bad epochs dropped
New count for 'Closed': 394
New count for 'Open': 394


<br><br>

<font size=6 color=#003366> <center><b>PART II - FEATURE GENERATION </b></center> </font> <br> <br>

<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">
    <font size=5><b> II.1. Create your own feature space <br></b></font>  
</div> <br>

<div class="alert alert-info"> <b>[Question 5] Create your feature space</b> <br> Your EEG segments are now ready for feature extraction. You must convert each 500 ms segment into a fixed-size feature vector suitable for classification. Multiple feature types exist: 
<ul>
    <li>Time-domain (e.g., mean, variance, kurtosis, etc.)</li>
    <li>Frequency-domain (e.g., band power in specific frequency bands, peak frequency, etc.)</li>
</ul>
Based on your observations of the EEG signals, which type(s) of features do you recommend using for this task? Why? Support your decision with signal plots, frequency content, or other justifications.
</div> 


<div class="alert alert-warning">
<b>Power of a frequency band </b>  <br>
To assist you, we provide a <samp>getBandPower</samp> function that calculates the power within specified frequency bands. You can modify this function as needed to suit your analysis and data formats. 
</div> 


This step is motivated by the known physiological difference between eyes open and eyes closed conditions: in particular, increased alpha-band activity (8–13 Hz) is typically observed when the eyes are closed. Therefore, spectral band power was chosen as the primary feature representation
EEG signals are inherently non-stationary and noisy in the time domain. However, their frequency content reflects meaningful neural oscillations associated with cognitive and physiological states. For this reason, frequency-domain analysis provides a more robust representation for classification tasks.
The resulting feature space explicitly encodes the most relevant physiological marker for this task: the increase in alpha-band power during eyes-closed conditions. By combining band-specific spectral information across channels, the model is provided with both spatial and frequency-domain structure, which is essential for accurate classification.

In [102]:
import numpy as np
from scipy.signal import welch
from scipy.integrate import simpson as simps

def getBandPower(signal, fs):
    #Compute relative band power of a signal using Simpson's rule.
    
    # Compute Power Spectral Density (PSD)
    # nfft=8192 provides high frequency resolution
    freqs, psd = welch(signal, fs=fs, nfft=8192)

    # Compute total power (area under the curve from 1 to 40Hz)
    total_idx = (freqs >= 1) & (freqs <= 40)
    total_power = simps(psd[total_idx], x=freqs[total_idx])

    # Define standard EEG frequency bands
    freqRanges = [
        (1, 4),   # Delta
        (4, 8),   # Theta
        (8, 13),  # Alpha (The most important for eye state!)
        (13, 30)  # Beta
    ]
    band_names = ['Delta', 'Theta', 'Alpha', 'Beta']

    band_powers = []
    for low, high in freqRanges:
        idx = (freqs >= low) & (freqs <= high)
        # Calculate area of the specific band
        band_power = simps(psd[idx], x=freqs[idx])
        # Normalize by total power to get RELATIVE power
        relative_power = band_power / (total_power + 1e-12)
        band_powers.append(relative_power)

    return band_powers, band_names

def extract_all_features(epochs, fs):
    #Iterates through all epochs and channels to build the feature matrix.
    # epochs.get_data() returns array of shape (n_epochs, n_channels, n_times)
    data = epochs.get_data(copy=False)
    n_epochs, n_channels, _ = data.shape
    
    features_list = []
    
    print(f"Extracting features from {n_epochs} segments...")
    
    for e in range(n_epochs): #loop throgh epochs 
        epoch_features = []
        for c in range(n_channels): #loop through channels 
            # Extract 4 bands per channel
            powers, names = getBandPower(data[e, c, :], fs) 
            epoch_features.extend(powers)
        
        features_list.append(epoch_features)
        
    return np.array(features_list)


# Extract features for both conditions
# X_closed will have shape (n_segments, n_channels * n_bands)
X_closed = extract_all_features(epochs_closed, fs)
X_open = extract_all_features(epochs_open, fs)

# Create target labels
# 0 for Eyes Closed, 1 for Eyes Open => binary classification
y_closed = np.zeros(len(X_closed))
y_open = np.ones(len(X_open))

# Concatenate into final dataset
X = np.vstack((X_closed, X_open)) #combine feature vectors
y = np.concatenate((y_closed, y_open)) #combine labels 

print("Feature Matrix X shape:", X.shape) # Should be (Total_Segments, 252)
print("Label Vector y shape:", y.shape)

Extracting features from 394 segments...
Extracting features from 394 segments...
Feature Matrix X shape: (788, 252)
Label Vector y shape: (788,)


<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">
    <font size=5><b> II.2. Dimensionality reduction <br></b></font>  
</div> <br>

<div class="alert alert-info">
<b>[Question 6] Reduce the number of features </b>  <br>
Implement either a feature selection or a feature extraction technique to reduce the number of features in you dataset.
</div> 

Before starting the data driven transformations, like PCA, the data set was split into a training and a test set. Because overlapping windows introduce strong correlations between neighboring epochs, the dataset was split using shuffled temporal blocks rather than a fully random split. This reduces information leakage while preserving a balanced distribution of signal characteristics between training and testing sets. 

In [103]:
from sklearn.model_selection import train_test_split
#Split data into Training and Testing sets (80/20 split)
#training set is used for model fitting and hyperparameter tuning, the test set is kept unseen until evaluation
# Consecutive overlapping epochs are highly correlated.
# Instead of randomly splitting individual epochs, we split larger temporal blocks.
block_size = 20
# Number of complete blocks
n_samples=len(X)
n_blocks=n_samples//block_size
# Create block indices
blocks = np.arange(n_blocks)
# Randomly split blocks, random_state can be any fixed variable as long as it is consistent
train_blocks, test_blocks = train_test_split(blocks, test_size=0.2, random_state=42,shuffle=True)
# Convert block indices into sample indices
train_idx = np.concatenate([np.arange(b * block_size, (b + 1) * block_size)for b in train_blocks])
test_idx = np.concatenate([np.arange(b * block_size, (b + 1) * block_size)for b in test_blocks])

# Create train/test sets
X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

The feature extraction step produced a high-dimensional feature space containing spectral features from all EEG channels. Although this representation preserves detailed spectral and spatial information, high-dimensional datasets may increase computational cost and lead to overfitting, especially when features are correlated. To address this issue, dimensionality reduction was performed using Principal Component Analysis (PCA). Standardization ensures that all features contribute equally to the decomposition process since, PCA is variance-based and therefore sensitive to feature scaling. This reduction provides several advantages as lower computational complexity, reduced feature redundancy, improved model generalization, reduced risk of overfitting. Because EEG band-power features are often highly correlated across neighboring channels and frequency bands, PCA is particularly suitable for compressing the information into a smaller and more informative representation.


In [104]:
# Insert your code here
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

#Standardization:
#PCA is variance based and features must have comparable scales, we apply zero mean and unit varaince normalization
#fit() learns the recipe, transform() cook using the recipe, fit_transform learn+cook in one step 

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca_full = PCA()
pca_full.fit(X_train_scaled)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1
print("Components for 90% variance:", n_components_90)

#PCA (fit ONLY on training data)
pca = PCA(n_components=n_components_90)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

#plots the two dimensions of the original feature space
plt.figure(figsize=(6,5))
plt.scatter(X_train_scaled[y_train == 0, 0],X_train_scaled[y_train == 0, 1],label='Closed',alpha=0.6)
plt.scatter(X_train_scaled[y_train == 1, 0],X_train_scaled[y_train == 1, 1],label='Open',alpha=0.6)
plt.title("Original Feature Space (First 2 Features)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

# Plotting first two dimensions of PCA
plt.scatter(X_train_pca[y_train==0, 0], X_train_pca[y_train==0, 1], label='Closed', alpha=0.6, c='navy')
plt.scatter(X_train_pca[y_train==1, 0], X_train_pca[y_train==1, 1], label='Open', alpha=0.6, c='darkorange')
plt.title("PCA Extraction (First 2 PCs)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()


plt.show()


Components for 90% variance: 64


<br><br>



<font size=6 color=#003366> <center><b>PART III - CLASSIFICATION </b></center> </font> <br> <br>





<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">

    <font size=5><b> III.1. Model and hyperparameter selection <br></b></font>  

</div> <br>



<div class="alert alert-info"> <b>[Question 7] Choose and evaluate your classifier</b> <br>

Now that you’ve defined your feature set, it’s time to evaluate whether those features are effective in distinguishing between "eyes open" and "eyes closed" EEG segments. Select at least two classifiers from the following list: K-nearest neighbors (kNN), Decision tree, Neural network, Support Vector Machine (SVM).  <br>





Use K-Fold cross-validation on the training data to find best hyperparametes for your models. Report the accuracy of each classifier and include basic performance comparisons.</li>



</div>

I chose to compare:
- Support Vector Machine (SVM): a margin-based classifier suitable for high-dimensional data
- k-Nearest Neighbors (kNN): a distance-based classifier that relies on local similarity

To ensure robust model evaluation, 5-fold cross-validation is applied within the training set. This method splits the training data into five subsets so in each iteration, four folds are used for training and one fold is used for validation. This process reduces the risk of overfitting and provides a more stable estimate of model performance compared to a single train-validation split.

For hyperparamter tuning i used gridsearch. 
For SVM, the following parameters are explored:
- C: regularization strength controlling model complexity
- kernel: linear, polynomial, and radial basis function (RBF)
- gamma: kernel coefficient for non-linear decision boundaries

For kNN:
- number of neighbors (k)
- distance metric (Euclidean, Manhattan and cosine)
- weighting strategy (uniform vs distance-based)

The best model is selected based on cross-validation accuracy.

In [105]:
# Insert your code here
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


#Setup Cross-Validation
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

#MODEL 1: SVM 
svm_param_grid = {'C': [0.1, 1, 10, 100],'kernel': ['rbf', 'linear','poly'],'degree':[1,2,3],'gamma': ['scale', 'auto']}
grid_svm = GridSearchCV(SVC(), svm_param_grid, cv=cv_strategy, scoring='accuracy')
grid_svm.fit(X_train_pca, y_train)

#MODEL 2: kNN
knn_param_grid = {'n_neighbors': [3, 5, 7, 11, 15], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan', 'cosine']}
grid_knn = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=cv_strategy, scoring='accuracy')
grid_knn.fit(X_train_pca, y_train)

# 3. Best Results Summary
print(f"Best SVM Params: {grid_svm.best_params_} | CV Accuracy: {grid_svm.best_score_:.3f}")
print(f"Best kNN Params: {grid_knn.best_params_} | CV Accuracy: {grid_knn.best_score_:.3f}")

Best SVM Params: {'C': 1, 'degree': 1, 'gamma': 'scale', 'kernel': 'rbf'} | CV Accuracy: 0.952
Best kNN Params: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'} | CV Accuracy: 0.897


The cross validation result shows that the SVM model achived the highest performance, which suggests that the feature space is well structured and supports effective decision boundary learning.


<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">
    <font size=5><b> III.2. Evaluate your best model <br></b></font>  
</div> <br>

   
<div class="alert alert-info">
<b>[Question 8] Performance on the test set </b>  <br>
 Once the best hyperparameters are selected, train your best model on the full training set with these hyperparameters, then evaluate on the test set to get an unbiased estimate of generalization performance. Comment
</div> 

After hyperparameter tuning, the models are evaluated on an independent test set. This test set is not used during training or cross-validation, ensuring an unbiased estimate of generalization performance.

Performance is measured using accuracy,precision, recall, and F1-score

In [106]:
from sklearn.metrics import accuracy_score, classification_report
# Evaluate SVM
y_pred_svm = grid_svm.predict(X_test_pca)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"Final Test Accuracy - SVM: {acc_svm:.3f}")
print("SVM Detailed Report:")
print(classification_report(y_test, y_pred_svm))
# Evaluate kNN
y_pred_knn = grid_knn.predict(X_test_pca)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f"Final Test Accuracy - kNN: {acc_knn:.3f}")
print("kNN Detailed Report:")
print(classification_report(y_test, y_pred_knn))


Final Test Accuracy - SVM: 0.912
SVM Detailed Report:
              precision    recall  f1-score   support

         0.0       0.90      0.87      0.88        60
         1.0       0.92      0.94      0.93       100

    accuracy                           0.91       160
   macro avg       0.91      0.90      0.91       160
weighted avg       0.91      0.91      0.91       160

Final Test Accuracy - kNN: 0.869
kNN Detailed Report:
              precision    recall  f1-score   support

         0.0       0.90      0.73      0.81        60
         1.0       0.86      0.95      0.90       100

    accuracy                           0.87       160
   macro avg       0.88      0.84      0.85       160
weighted avg       0.87      0.87      0.87       160



Although both models perform well, SVM is preferred due to its robustness in high-dimensional feature spaces and its ability to construct optimal decision boundaries. kNN, while simpler and interpretable, is more sensitive to noise and local variations in the dataset.

Analysis of the confusion matrix indicated that the SVM was slightly more sensitive to the EC condition, likely due to the highly distinct spectral signature of the alpha rhythm. Most misclassifications occurred during the EO condition, where frontal muscle artifacts occasionally mimicked high-frequency components or suppressed the global power distribution used for normalization.

Small variations in cross-validation scores and selected hyperparameters were observed between executions. This is expected due to the stochastic nature of data partitioning and the relatively limited EEG dataset size. Despite these variations, SVM consistently achieved higher cross-validation performance than kNN, indicating stable overall model behavior.

Overall, results confirm that EEG spectral features, provide strong discriminative power for distinguishing eye states.